In [1]:
!pip install -q autogluon.tabular scikit-learn==1.5.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.6/227.6 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.9/98.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 MB 39.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.22.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
datasets 4.4.2 requires pyarrow>=21.0.0, but you have pyarrow 20.0.0 which is incompatible.
category-encoders 2.9.0 requires scikit-learn>=1.6.0, but you have sciki

In [2]:
import warnings
import pandas as pd
from sklearn.metrics import roc_auc_score
from autogluon.tabular import TabularPredictor
from sklearn.preprocessing import LabelEncoder
import numpy as np
from sklearn.preprocessing import KBinsDiscretizer

warnings.filterwarnings('ignore')
RANDOM_STATE = 42

In [ ]:
TARGET = 'Churn'
TIME_LIMIT = 3600*9 
# TIME_LIMIT = 60

In [ ]:
# 2. Tinh chỉnhchỉnh
# ------------------------------------------------------------
class CFG:
    TARGET = 'Churn'
    N_FOLDS = 20
    INNER_FOLDS = 5
    RANDOM_SEED = 42

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e3/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e3/test.csv"
    ORIGINAL_PATH = "/kaggle/input/datasets/cdeotte/s6e3-original-dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv"

    # switches
    USE_CONSERVATIVE_FLAGS = True
    USE_SAFE_DIRECTED_BIGRAM = True
    USE_SAFE_ENGINEERED_ORIG_PRIORS = True
    USE_SAFE_DIRECTED_ORIG_PRIORS = True
    REMOVE_CONSTANT_NUMERIC_COLS = True
    REMOVE_EXACT_DUPLICATE_NUMERIC_COLS = True

    # Pseudo labels
    PSEUDO_LABELS = True
    TRES = 0.999

TOP_CATS_FOR_NGRAM = [
    'Contract', 'InternetService', 'PaymentMethod',
    'OnlineSecurity', 'TechSupport', 'PaperlessBilling'
]

In [5]:
# 3. Load Data
# ------------------------------------------------------------
print("Loading datasets...")
train = pd.read_csv(CFG.TRAIN_PATH)
test = pd.read_csv(CFG.TEST_PATH)
orig = pd.read_csv(CFG.ORIGINAL_PATH)

train[CFG.TARGET] = train[CFG.TARGET].map({'No': 0, 'Yes': 1}).astype(int)
orig[CFG.TARGET] = orig[CFG.TARGET].map({'No': 0, 'Yes': 1}).astype(int)

orig['TotalCharges'] = pd.to_numeric(orig['TotalCharges'], errors='coerce')
orig['TotalCharges'].fillna(orig['TotalCharges'].median(), inplace=True)

if 'customerID' in orig.columns:
    orig.drop(columns=['customerID'], inplace=True)

train_ids = train['id'].copy()
test_ids = test['id'].copy()

print(f"Train Shape : {train.shape}")
print(f"Test Shape  : {test.shape}")
print(f"Original Shape: {orig.shape}")

Loading datasets...
Train Shape : (594194, 21)
Test Shape  : (254655, 20)
Original Shape: (7043, 20)


In [6]:
from itertools import combinations
from collections import defaultdict

In [ ]:
# 4. Feature Engineering
# ------------------------------------------------------------
CATS = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod'
]
NUMS = ['tenure', 'MonthlyCharges', 'TotalCharges']

NEW_NUMS = []
NEW_CATS = []
NUM_AS_CAT = []

print("Feature Engineering Pipeline Started...")
print("=" * 60)

# [1/8] Tính toán số lần xuất hiện của mỗi giá trị trong mỗi cột số học
print("[1/8] Creating Frequency Encoding features...")
for col in NUMS:
    freq = pd.concat([train[col], orig[col], test[col]]).value_counts(normalize=True)
    for df in [train, test, orig]:
        df[f'FREQ_{col}'] = df[col].map(freq).fillna(0).astype('float32')
    NEW_NUMS.append(f'FREQ_{col}')
print(f"    Created {len(NUMS)} frequency features")

# [2/8] Tương tác số học
print("[2/8] Creating Arithmetic Interaction features...")
for df in [train, test, orig]:
    df['charges_deviation'] = (df['TotalCharges'] - df['tenure'] * df['MonthlyCharges']).astype('float32')
    df['monthly_to_total_ratio'] = (df['MonthlyCharges'] / (df['TotalCharges'] + 1)).astype('float32')
    df['avg_monthly_charges'] = (df['TotalCharges'] / (df['tenure'] + 1)).astype('float32')
NEW_NUMS += ['charges_deviation', 'monthly_to_total_ratio', 'avg_monthly_charges']
print("    Created 3 arithmetic features")

# [3/8] Service Counts + Structured Service Features
print("[3/8] Creating Service Count features...")
SERVICE_COLS = [
    'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'
]
SUPPORT_COLS = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']
STREAMING_COLS = ['StreamingTV', 'StreamingMovies']
ADDON_COLS = [
    'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'
]

for df in [train, test, orig]:
    df['service_count'] = (df[SERVICE_COLS] == 'Yes').sum(axis=1).astype('float32')
    df['support_count'] = (df[SUPPORT_COLS] == 'Yes').sum(axis=1).astype('float32')
    df['streaming_count'] = (df[STREAMING_COLS] == 'Yes').sum(axis=1).astype('float32')
    df['addon_count'] = (df[ADDON_COLS] == 'Yes').sum(axis=1).astype('float32')
    df['has_internet'] = (df['InternetService'] != 'No').astype('float32')
    df['has_phone'] = (df['PhoneService'] == 'Yes').astype('float32')
    df['service_count_bin'] = pd.cut(
        df['service_count'],
        bins=[-1, 1, 3, 5, 100],
        labels=['0_1', '2_3', '4_5', '6_plus']
    )
    df['paperless_echeck'] = (
        (df['PaperlessBilling'] == 'Yes') &
        (df['PaymentMethod'] == 'Electronic check')
    ).astype('float32')
    df['monthly_per_service'] = (
        df['MonthlyCharges'] / (df['service_count'] + 1.0)
    ).astype('float32')

    if CFG.USE_CONSERVATIVE_FLAGS:
        df['is_month_to_month'] = (df['Contract'] == 'Month-to-month').astype('float32')
        df['is_fiber'] = (df['InternetService'] == 'Fiber optic').astype('float32')
        df['is_electronic_check'] = (df['PaymentMethod'] == 'Electronic check').astype('float32')
        df['contract_term_months'] = df['Contract'].map({
            'Month-to-month': 1,
            'One year': 12,
            'Two year': 24
        }).fillna(0).astype('float32')

NEW_NUMS += [
    'service_count', 'has_internet', 'has_phone',
    'support_count', 'streaming_count', 'addon_count',
    'paperless_echeck', 'monthly_per_service'
]
if CFG.USE_CONSERVATIVE_FLAGS:
    NEW_NUMS += [
        'is_month_to_month', 'is_fiber',
        'is_electronic_check', 'contract_term_months'
    ]
NEW_CATS += ['service_count_bin']
print("    Created base + structured service features")

# [4/8] Price Pressure + Binned Regimes
print("[4/8] Creating price pressure and binned regime features...")
for df in [train, test, orig]:
    df['charge_gap_current_vs_avg'] = (
        df['MonthlyCharges'] - df['avg_monthly_charges']
    ).astype('float32')

    df['m2m_monthly_pressure'] = (
        df['MonthlyCharges'] * (df['Contract'] == 'Month-to-month').astype('float32')
    ).astype('float32')

    df['tenure_monthly_interaction'] = (
        df['tenure'] * df['MonthlyCharges']
    ).astype('float32')

    df['tenure_bin'] = pd.cut(
        df['tenure'],
        bins=[-1, 1, 6, 12, 24, 48, 72],
        labels=['0_1', '2_6', '7_12', '13_24', '25_48', '49_72']
    )

    df['monthly_bin'] = pd.cut(
        df['MonthlyCharges'],
        bins=[-np.inf, 35, 70, 90, np.inf],
        labels=['low', 'mid', 'high', 'very_high']
    )

NEW_NUMS += [
    'charge_gap_current_vs_avg',
    'm2m_monthly_pressure',
    'tenure_monthly_interaction'
]
NEW_CATS += ['tenure_bin', 'monthly_bin']
print("    Created charge/pressure features and bins")

# [5/8] ORIG_proba Mapping
print("[5/8] Creating ORIG_proba features (target probability from original data)...")
for col in CATS + NUMS:
    tmp = orig.groupby(col)[CFG.TARGET].mean()
    _name = f"ORIG_proba_{col}"
    train = train.merge(tmp.rename(_name), on=col, how="left")
    test = test.merge(tmp.rename(_name), on=col, how="left")
    for df in [train, test]:
        df[_name] = df[_name].fillna(0.5).astype('float32')
    NEW_NUMS.append(_name)
print(f"    Created {len(CATS + NUMS)} ORIG_proba features")

# [6/8] Đặc điểm phân phối
print("[6/8] Creating Distribution Features (percentile ranks, z-scores)...")
def pctrank_against(values, reference):
    ref_sorted = np.sort(reference)
    return (np.searchsorted(ref_sorted, values) / len(ref_sorted)).astype('float32')

def zscore_against(values, reference):
    mu, sigma = np.mean(reference), np.std(reference)
    return (np.zeros(len(values), dtype='float32') if sigma == 0
            else ((values - mu) / sigma).astype('float32'))

orig_churner_tc = orig.loc[orig[CFG.TARGET] == 1, 'TotalCharges'].values
orig_nonchurner_tc = orig.loc[orig[CFG.TARGET] == 0, 'TotalCharges'].values
orig_tc = orig['TotalCharges'].values
orig_is_mc_mean = orig.groupby('InternetService')['MonthlyCharges'].mean()

for df in [train, test]:
    tc = df['TotalCharges'].values
    df['pctrank_nonchurner_TC'] = pctrank_against(tc, orig_nonchurner_tc)
    df['pctrank_churner_TC'] = pctrank_against(tc, orig_churner_tc)
    df['pctrank_orig_TC'] = pctrank_against(tc, orig_tc)
    df['zscore_churn_gap_TC'] = (
        np.abs(zscore_against(tc, orig_churner_tc)) -
        np.abs(zscore_against(tc, orig_nonchurner_tc))
    ).astype('float32')
    df['zscore_nonchurner_TC'] = zscore_against(tc, orig_nonchurner_tc)
    df['pctrank_churn_gap_TC'] = (
        pctrank_against(tc, orig_churner_tc) -
        pctrank_against(tc, orig_nonchurner_tc)
    ).astype('float32')
    df['resid_IS_MC'] = (
        df['MonthlyCharges'] - df['InternetService'].map(orig_is_mc_mean).fillna(0)
    ).astype('float32')

    vals = np.zeros(len(df), dtype='float32')
    for cat_val in orig['InternetService'].unique():
        mask = df['InternetService'] == cat_val
        ref = orig.loc[orig['InternetService'] == cat_val, 'TotalCharges'].values
        if len(ref) > 0 and mask.sum() > 0:
            vals[mask] = pctrank_against(df.loc[mask, 'TotalCharges'].values, ref)
    df['cond_pctrank_IS_TC'] = vals

    vals = np.zeros(len(df), dtype='float32')
    for cat_val in orig['Contract'].unique():
        mask = df['Contract'] == cat_val
        ref = orig.loc[orig['Contract'] == cat_val, 'TotalCharges'].values
        if len(ref) > 0 and mask.sum() > 0:
            vals[mask] = pctrank_against(df.loc[mask, 'TotalCharges'].values, ref)
    df['cond_pctrank_C_TC'] = vals

DIST_FEATURES = [
    'pctrank_nonchurner_TC', 'zscore_churn_gap_TC', 'pctrank_churn_gap_TC',
    'resid_IS_MC', 'cond_pctrank_IS_TC', 'zscore_nonchurner_TC',
    'pctrank_orig_TC', 'pctrank_churner_TC', 'cond_pctrank_C_TC'
]
NEW_NUMS += DIST_FEATURES
print(f"    Created {len(DIST_FEATURES)} distribution features")

# [7/8] Đặc điểm khoảng cách lượng tử
print("[7/8] Creating Quantile Distance Features...")
for q_label, q_val in [('q25', 0.25), ('q50', 0.50), ('q75', 0.75)]:
    ch_q = np.quantile(orig_churner_tc, q_val)
    nc_q = np.quantile(orig_nonchurner_tc, q_val)
    for df in [train, test]:
        df[f'dist_To_ch_{q_label}'] = np.abs(df['TotalCharges'] - ch_q).astype('float32')
        df[f'dist_To_nc_{q_label}'] = np.abs(df['TotalCharges'] - nc_q).astype('float32')
        df[f'qdist_gap_To_{q_label}'] = (
            df[f'dist_To_nc_{q_label}'] - df[f'dist_To_ch_{q_label}']
        ).astype('float32')

QDIST_FEATURES = [
    'qdist_gap_To_q50', 'dist_To_ch_q50', 'dist_To_nc_q50',
    'dist_To_nc_q25', 'qdist_gap_To_q25',
    'dist_To_nc_q75', 'dist_To_ch_q75', 'qdist_gap_To_q75'
]
NEW_NUMS += QDIST_FEATURES
print(f"    Created {len(QDIST_FEATURES)} quantile distance features")

# [8/8] Numericals as Categories
print("[8/8] Creating Numericals-as-Categories features...")
for col in NUMS:
    _new = f'CAT_{col}'
    NUM_AS_CAT.append(_new)
    for df in [train, test]:
        df[_new] = df[col].astype(str).astype('category')
print(f"    Created {len(NUMS)} numerical-as-category features")

# Digit Features
DIGIT_FEATURES = [
    'tenure_first_digit','tenure_last_digit','tenure_second_digit',
    'tenure_mod10','tenure_mod12','tenure_num_digits',
    'tenure_is_multiple_10','tenure_rounded_10','tenure_dev_from_round10',
    'mc_first_digit','mc_last_digit','mc_second_digit',
    'mc_mod10','mc_mod100','mc_num_digits',
    'mc_is_multiple_10','mc_is_multiple_50',
    'mc_rounded_10','mc_fractional','mc_dev_from_round10',
    'tc_first_digit','tc_last_digit','tc_second_digit',
    'tc_mod10','tc_mod100','tc_num_digits',
    'tc_is_multiple_10','tc_is_multiple_100',
    'tc_rounded_100','tc_fractional','tc_dev_from_round100',
    'tenure_years','tenure_months_in_year',
    'mc_per_digit','tc_per_digit'
]

for df in [train, test]:
    t_str = df['tenure'].astype(str)
    df['tenure_first_digit'] = t_str.str[0].astype(int)
    df['tenure_last_digit'] = t_str.str[-1].astype(int)
    df['tenure_second_digit'] = t_str.apply(lambda x: int(x[1]) if len(x) > 1 else 0)
    df['tenure_mod10'] = df['tenure'] % 10
    df['tenure_mod12'] = df['tenure'] % 12
    df['tenure_num_digits'] = t_str.str.len()
    df['tenure_is_multiple_10'] = (df['tenure'] % 10 == 0).astype('float32')
    df['tenure_rounded_10'] = np.round(df['tenure'] / 10) * 10
    df['tenure_dev_from_round10'] = abs(df['tenure'] - df['tenure_rounded_10'])

    mc_str = df['MonthlyCharges'].astype(str).str.replace('.', '', regex=False)
    df['mc_first_digit'] = mc_str.str[0].astype(int)
    df['mc_last_digit'] = mc_str.str[-1].astype(int)
    df['mc_second_digit'] = mc_str.apply(lambda x: int(x[1]) if len(x) > 1 else 0)
    df['mc_mod10'] = np.floor(df['MonthlyCharges']) % 10
    df['mc_mod100'] = np.floor(df['MonthlyCharges']) % 100
    df['mc_num_digits'] = np.floor(df['MonthlyCharges']).astype(int).astype(str).str.len()
    df['mc_is_multiple_10'] = (np.floor(df['MonthlyCharges']) % 10 == 0).astype('float32')
    df['mc_is_multiple_50'] = (np.floor(df['MonthlyCharges']) % 50 == 0).astype('float32')
    df['mc_rounded_10'] = np.round(df['MonthlyCharges'] / 10) * 10
    df['mc_fractional'] = df['MonthlyCharges'] - np.floor(df['MonthlyCharges'])
    df['mc_dev_from_round10'] = abs(df['MonthlyCharges'] - df['mc_rounded_10'])

    tc_str = df['TotalCharges'].astype(str).str.replace('.', '', regex=False)
    df['tc_first_digit'] = tc_str.str[0].astype(int)
    df['tc_last_digit'] = tc_str.str[-1].astype(int)
    df['tc_second_digit'] = tc_str.apply(lambda x: int(x[1]) if len(x) > 1 else 0)
    df['tc_mod10'] = np.floor(df['TotalCharges']) % 10
    df['tc_mod100'] = np.floor(df['TotalCharges']) % 100
    df['tc_num_digits'] = np.floor(df['TotalCharges']).astype(int).astype(str).str.len()
    df['tc_is_multiple_10'] = (np.floor(df['TotalCharges']) % 10 == 0).astype('float32')
    df['tc_is_multiple_100'] = (np.floor(df['TotalCharges']) % 100 == 0).astype('float32')
    df['tc_rounded_100'] = np.round(df['TotalCharges'] / 100) * 100
    df['tc_fractional'] = df['TotalCharges'] - np.floor(df['TotalCharges'])
    df['tc_dev_from_round100'] = abs(df['TotalCharges'] - df['tc_rounded_100'])

    df['tenure_years'] = df['tenure'] // 12
    df['tenure_months_in_year'] = df['tenure'] % 12
    df['mc_per_digit'] = df['MonthlyCharges'] / (df['mc_num_digits'] + 0.001)
    df['tc_per_digit'] = df['TotalCharges'] / (df['tc_num_digits'] + 0.001)

NEW_NUMS += DIGIT_FEATURES
print(f"    Created {len(DIGIT_FEATURES)} digital numerical features")

# Composite Categorical Features
print("\n" + "=" * 60)
print("Creating Bi-gram / Tri-gram Composite Features...")
print("=" * 60)

BIGRAM_COLS = []
TRIGRAM_COLS = []

print(f"\nCreating Bi-gram features from top {len(TOP_CATS_FOR_NGRAM)} categoricals:")
print(f"  Columns: {TOP_CATS_FOR_NGRAM}")
for c1, c2 in combinations(TOP_CATS_FOR_NGRAM, 2):
    col_name = f"BG_{c1}_{c2}"
    for df in [train, test, orig]:
        df[col_name] = (df[c1].astype(str) + "_" + df[c2].astype(str)).astype('category')
    BIGRAM_COLS.append(col_name)

print(f"  Created {len(BIGRAM_COLS)} bi-gram features")

TOP4 = TOP_CATS_FOR_NGRAM[:4]
print(f"\nCreating Tri-gram features from top 4 categoricals:")
print(f"  Columns: {TOP4}")
for c1, c2, c3 in combinations(TOP4, 3):
    col_name = f"TG_{c1}_{c2}_{c3}"
    for df in [train, test, orig]:
        df[col_name] = (
            df[c1].astype(str) + "_" + df[c2].astype(str) + "_" + df[c3].astype(str)
        ).astype('category')
    TRIGRAM_COLS.append(col_name)

print(f"  Created {len(TRIGRAM_COLS)} tri-gram features")

print("\nCreating manual directed interaction features...")
MANUAL_BIGRAMS = [
    ('tenure_bin', 'Contract'),
    ('service_count_bin', 'Contract')
]
if CFG.USE_SAFE_DIRECTED_BIGRAM:
    MANUAL_BIGRAMS.append(('monthly_bin', 'Contract'))

MANUAL_TRIGRAMS = [
    ('tenure_bin', 'Contract', 'InternetService')
]

for c1, c2 in MANUAL_BIGRAMS:
    col_name = f"BG_{c1}_{c2}"
    for df in [train, test, orig]:
        df[col_name] = (df[c1].astype(str) + "_" + df[c2].astype(str)).astype('category')
    BIGRAM_COLS.append(col_name)

for c1, c2, c3 in MANUAL_TRIGRAMS:
    col_name = f"TG_{c1}_{c2}_{c3}"
    for df in [train, test, orig]:
        df[col_name] = (
            df[c1].astype(str) + "_" + df[c2].astype(str) + "_" + df[c3].astype(str)
        ).astype('category')
    TRIGRAM_COLS.append(col_name)

print(f"  Total bi-gram features:  {len(BIGRAM_COLS)}")
print(f"  Total tri-gram features: {len(TRIGRAM_COLS)}")

NGRAM_COLS = BIGRAM_COLS + TRIGRAM_COLS
print(f"\nTotal composite features: {len(NGRAM_COLS)}")

# Ưu tiên ORIG an toàn trên các tính năng được thiết kế cho thẻ có dung lượng thấp
# ------------------------------------------------------------
print("\nCreating safe ORIG priors for engineered low-card features...")
safe_prior_cols = []
if CFG.USE_SAFE_ENGINEERED_ORIG_PRIORS:
    safe_prior_cols += [
        'service_count', 'support_count', 'streaming_count', 'addon_count',
        'paperless_echeck', 'service_count_bin', 'tenure_bin', 'monthly_bin'
    ]
    if CFG.USE_CONSERVATIVE_FLAGS:
        safe_prior_cols += [
            'contract_term_months', 'is_month_to_month', 'is_fiber', 'is_electronic_check'
        ]

safe_prior_cols = [c for c in safe_prior_cols if c in orig.columns]
created_safe_orig_priors = []

for col in safe_prior_cols:
    tmp = orig.groupby(col, observed=False)[CFG.TARGET].mean()
    _name = f"ORIG_proba_SAFE_{col}"
    train = train.merge(tmp.rename(_name), on=col, how='left')
    test = test.merge(tmp.rename(_name), on=col, how='left')
    for df in [train, test]:
        df[_name] = df[_name].fillna(0.5).astype('float32')
    NEW_NUMS.append(_name)
    created_safe_orig_priors.append(_name)

safe_directed_prior_cols = []
if CFG.USE_SAFE_DIRECTED_ORIG_PRIORS:
    safe_directed_prior_cols = [
        'BG_tenure_bin_Contract',
        'BG_monthly_bin_Contract',
        'BG_service_count_bin_Contract'
    ]

created_safe_directed_priors = []
for col in safe_directed_prior_cols:
    if col not in orig.columns:
        continue
    tmp = orig.groupby(col, observed=False)[CFG.TARGET].mean()
    _name = f"ORIG_proba_SAFE_{col}"
    train = train.merge(tmp.rename(_name), on=col, how='left')
    test = test.merge(tmp.rename(_name), on=col, how='left')
    for df in [train, test]:
        df[_name] = df[_name].fillna(0.5).astype('float32')
    NEW_NUMS.append(_name)
    created_safe_directed_priors.append(_name)

print(f"    Created {len(created_safe_orig_priors)} engineered ORIG prior features")
print(f"    Created {len(created_safe_directed_priors)} directed-interaction ORIG prior features")

# Kiểm tra tính dư thừa số học an toàn: chỉ hằng số + các bản sao chính xác
# ------------------------------------------------------------
print("\nRunning safe redundancy checks on numeric features...")
numeric_pool = []
seen_numeric = set()
for c in NUMS + NEW_NUMS:
    if c not in seen_numeric and c in train.columns and c in test.columns:
        numeric_pool.append(c)
        seen_numeric.add(c)

const_drop = []
if CFG.REMOVE_CONSTANT_NUMERIC_COLS:
    for c in numeric_pool:
        if train[c].nunique(dropna=False) <= 1 and test[c].nunique(dropna=False) <= 1:
            const_drop.append(c)

dup_drop = []
if CFG.REMOVE_EXACT_DUPLICATE_NUMERIC_COLS:
    hash_buckets = defaultdict(list)
    for c in numeric_pool:
        if c in const_drop:
            continue
        train_hash = int(pd.util.hash_pandas_object(train[c], index=False).sum())
        test_hash = int(pd.util.hash_pandas_object(test[c], index=False).sum())
        hash_buckets[(train_hash, test_hash)].append(c)

    for _, cols in hash_buckets.items():
        if len(cols) <= 1:
            continue
        base = cols[0]
        for c in cols[1:]:
            if train[base].equals(train[c]) and test[base].equals(test[c]):
                dup_drop.append(c)

drop_numeric = sorted(set(const_drop + dup_drop))
if drop_numeric:
    NEW_NUMS = [c for c in NEW_NUMS if c not in drop_numeric]
    print(f"    Dropped {len(drop_numeric)} safe redundant numeric features")
    print("    Removed:", drop_numeric)
else:
    print("    No constant/exact-duplicate numeric features found")

# Tóm tắt tính năng
FEATURES = NUMS + CATS + NEW_NUMS + NUM_AS_CAT + NEW_CATS + NGRAM_COLS
TE_COLUMNS = NUM_AS_CAT + CATS + NEW_CATS
TE_NGRAM_COLUMNS = NGRAM_COLS
TO_REMOVE = NUM_AS_CAT + CATS + NEW_CATS + NGRAM_COLS
STATS = ['std', 'min', 'max']

print("\n" + "=" * 60)
print("FEATURE ENGINEERING COMPLETE")
print("=" * 60)
print(f"Total Features: {len(FEATURES)}")
print(f"  - Numerical:             {len(NUMS)}")
print(f"  - Categorical:           {len(CATS)}")
print(f"  - Engineered Numerical:  {len(NEW_NUMS)}")
print(f"  - Num-as-Cat:            {len(NUM_AS_CAT)}")
print(f"  - Added Categorical:     {len(NEW_CATS)}")
print(f"  - Bi-gram:               {len(BIGRAM_COLS)}")
print(f"  - Tri-gram:              {len(TRIGRAM_COLS)}")

Feature Engineering Pipeline Started...
[1/8] Creating Frequency Encoding features...
    Created 3 frequency features
[2/8] Creating Arithmetic Interaction features...
    Created 3 arithmetic features
[3/8] Creating Service Count features...
    Created base + structured service features
[4/8] Creating price pressure and binned regime features...
    Created charge/pressure features and bins
[5/8] Creating ORIG_proba features (target probability from original data)...
    Created 19 ORIG_proba features
[6/8] Creating Distribution Features (percentile ranks, z-scores)...
    Created 9 distribution features
[7/8] Creating Quantile Distance Features...
    Created 8 quantile distance features
[8/8] Creating Numericals-as-Categories features...
    Created 3 numerical-as-category features
    Created 35 digital numerical features

Creating Bi-gram / Tri-gram Composite Features...

Creating Bi-gram features from top 6 categoricals:
  Columns: ['Contract', 'InternetService', 'PaymentMethod

In [8]:
for c in CATS + NEW_CATS + NUM_AS_CAT + BIGRAM_COLS + TRIGRAM_COLS:
    train[c] = train[c].astype(str).astype('category')
    test[c] = test[c].astype(str).astype('category')

# Training with AutoGluon

In [9]:
X = train[FEATURES + [CFG.TARGET]]
X_test = test[FEATURES]

In [10]:
predictor = TabularPredictor(
    label=TARGET, 
    problem_type='binary',
    eval_metric='roc_auc'
).fit(
    train_data=X,
    presets='best_quality_v150',
    time_limit=TIME_LIMIT,
    verbosity=2
)

No path specified. Models will be saved in: "AutogluonModels/ag-20260327_185856"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Sat Jan 17 11:20:45 UTC 2026
CPU Count:          4
Pytorch Version:    2.8.0+cu126
CUDA Version:       CUDA is not available
Memory Avail:       28.90 GB / 31.35 GB (92.2%)
Disk Space Avail:   19.50 GB / 19.52 GB (99.9%)
Presets specified: ['best_quality_v150']
Using hyperparameters preset: hyperparameters='zeroshot_2025_12_18_cpu'
Stack configuration (auto_stack=True): num_stack_levels=0, num_bag_folds=8, num_bag_sets=1
Beginning AutoGluon training ... Time limit = 32400s
AutoGluon will save models to "/kaggle/working/AutogluonModels/ag-20260327_185856"
Train Data Rows:    594194
Train Data Columns: 152
Label Column:       Churn
Problem Type:       binary
Preprocessing data ...
Selected clas

# Leaderboard and Performance (OOF)

In [11]:
predictor.leaderboard(silent=True).style.background_gradient(subset=['score_val'], cmap='RdYlGn') 

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,0.918969,roc_auc,66.261113,31750.203486,0.135578,10.435915,2,True,4
1,CatBoost_c1_BAG_L1,0.918878,roc_auc,14.751262,31060.927030,14.751262,31060.927030,1,True,1
2,LightGBM_r177_BAG_L1,0.917064,roc_auc,51.374273,678.840541,51.374273,678.840541,1,True,2
3,LightGBM_r163_BAG_L1,0.914780,roc_auc,22.517808,492.158067,22.517808,492.158067,1,True,3


In [12]:
oof_predictions = predictor.predict_proba_oof()[1] 
oof_roc_auc = roc_auc_score(train[TARGET], oof_predictions) 
print("-----------------------") 
print(f"Overall Out-of-Fold (OOF) Roc Auc Score: {oof_roc_auc:.5f}")

oof_df = pd.DataFrame({'id': train['id'], TARGET: oof_predictions.values})
oof_df.to_csv('oof.csv', index=False) 
oof_df

-----------------------
Overall Out-of-Fold (OOF) Roc Auc Score: 0.91897


,id,Churn
0,0,0.008109
1,1,0.002974
2,2,0.302790
3,3,0.711518
4,4,0.777315
...,...,...
594189,594189,0.024662
594190,594190,0.004777
594191,594191,0.003039
594192,594192,0.526768


# Predictions and Submission

In [13]:
test_preds = predictor.predict_proba(X_test)[1] 

submission = pd.DataFrame({
    'id': test['id'], 
    TARGET: test_preds.values
})

submission.to_csv('submission.csv', index=False)
submission

,id,Churn
0,594194,0.106300
1,594195,0.001083
2,594196,0.125133
3,594197,0.003147
4,594198,0.563932
...,...,...
254650,848844,0.001109
254651,848845,0.777047
254652,848846,0.289355
254653,848847,0.002750
